# Paradesic LLM RAG system

***
## Introduction

### All the Fields on an `Item`

So far we've only queried `full_identifier` and `title`, which is fairly rudimentary. Real research questions usually need much more - which languages are involved, which collection an item belongs to, which university holds it, links to the actual recordings, and so on. Since the *schema* fully defines the `Item` *type*, we can see every *field* available to us, organized here by whether each one is a *Scalar* (a leaf value) or an *Object* / list (which needs its own sub-*field*s).

**Scalar fields** (query these directly, no sub-fields needed):

| Field | Type |
|---|---|
| `access_class` | `String` |
| `access_condition_name` | `String` |
| `access_narrative` | `String` |
| `born_digital` | `Boolean` |
| `citation` | `String` |
| `created_at` | `ISO8601DateTime` |
| `description` | `String` |
| `dialect` | `String` |
| `digitised_on` | `String` |
| `doi` | `String` |
| `doi_json` | `String` |
| `essences_count` | `Int` |
| `full_identifier` | `String!` |
| `id` | `ID!` |
| `identifier` | `String!` |
| `ingest_notes` | `String` |
| `language` | `String` |
| `metadata_exportable` | `Boolean!` |
| `original_media` | `String` |
| `originated_on` | `String` |
| `originated_on_narrative` | `String` |
| `permalink` | `String!` |
| `private` | `Boolean` |
| `public` | `Boolean` |
| `received_on` | `String` |
| `region` | `String` |
| `title` | `String` |
| `tracking` | `String` |
| `updated_at` | `ISO8601DateTime` |

**Object fields** (a single related *Object* - needs its own `{ }` block with sub-*field*s):

| Field | Type |
|---|---|
| `access_condition` | `AccessCondition` |
| `boundaries` | `Boundary` |
| `collection` | `Collection!` |
| `collector` | `Person!` |
| `discourse_type` | `DiscourseType` |
| `operator` | `Person` |
| `university` | `University` |

**List fields** (a list of related *Object*s - also needs its own `{ }` block with sub-*field*s):

| Field | Type |
|---|---|
| `content_languages` | `[Language]` |
| `countries` | `[Country]` |
| `data_categories` | `[DataCategory]` |
| `data_types` | `[DataType]` |
| `essences` | `[Essence]` |
| `item_agents` | `[Agent]` |
| `subject_languages` | `[Language]` |

Putting this together, we can write a much richer query than our original example:

```json
{
  items(full_identifier: "AA1-001") {
    results {
      full_identifier
      title
      description
      region
      collection {
        title
      }
      collector {
        name
      }
      university {
        name
      }
      essences {
        filename
        permalink
      }
      content_languages {
        name
      }
    }
  }
}
```

None of this field-by-field detail is written up anywhere in prose - it only exists by reading the *schema* itself, which is exactly the kind of "documentation via schema" we discussed above.

***
## Accessing an API

Now we've looked through **PARADISEC**'s *schema* and should be ready to start making our own queries. In this notebook, we'll be using [Python](https://www.python.org/) to do so. However, we once again have to take a step back - how do we actually communicate with an **API** in the first place?

No matter the kind, every **API** has at least one *endpoint*. In this context, an *endpoint* is simply the URL where an **API** can be accessed. It's possible to have multiple *endpoint*s that expose different resources/functions. For a **GraphQL** **API**, because the different functions are covered by the various *field*s that fall under `Query`, there is typically only one *endpoint*. Let's store the *endpoint* for **PARADISEC**'s **GraphQL** **API** below.

In [1]:
API_URL = 'https://admin-catalog.paradisec.org.au/graphql'

To actually interact with web resources, we need the ability to send HTTP requests. There's a python library that makes this process fairly simple, called [Requests](https://docs.python-requests.org/en/latest/index.html); let's import that library below.

In [2]:
import requests
import getpass

/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [ ]:
email = input('Email:')
password = getpass.getpass('Password:')

Now that we have login information, we can program that extra piece we need. The code is below, followed by a basic explanation of each part.

In [ ]:
from bs4 import BeautifulSoup

session = requests.Session()

login_page = session.get('https://admin-catalog.paradisec.org.au/users/sign_in')
soup = BeautifulSoup(login_page.text, 'html.parser')
csrf = soup.find('input', {'name': 'authenticity_token'})['value']

response = session.post(
    'https://admin-catalog.paradisec.org.au/users/sign_in',
    data={
        'authenticity_token': csrf,
        'user[email]': email,
        'user[password]': password,
    }
)

try:
    response.raise_for_status()
    print('Login Successful!')
except: print('Login Failed...')

***
## Making a Query

Let's try running the example query one more time! This time, however, we'll make the request through the session we set up.

In [5]:
query = '''
    {
        items(full_identifier: "ABC") {
            total
            next_page
            results {
                full_identifier
                title
            }
        }
    }
'''
response = session.post(API_URL, json={'query': query})

response.json()

{'data': {'items': {'total': 1,
   'next_page': None,
   'results': [{'full_identifier': 'WD1-ABC',
     'title': 'Tum-why-village-moved-here'}]}}}

In [6]:
print(response.json()['data']['items']['total'])
print(response.json()['data']['items']['next_page'])

for result in response.json()['data']['items']['results']:
    print(result['full_identifier'])
    print(result['title'])

1
None
WD1-ABC
Tum-why-village-moved-here


***
## Building a Local Text Corpus: NT1 Collection

Now we'll put everything we've learned to use to build the first stage of a **RAG** (Retrieval-Augmented Generation) pipeline over PARADISEC texts. That pipeline has four stages:

1. **Extract** text from selected item fields and linked objects
2. **Persist** the extracted texts to local files, along with a CSV that tracks collection ID, full identifier, sub-item (essence) fields, and the text itself
3. **Vectorize** those texts into a Chroma vector database
4. Implement **natural-language search** (RAG) over that database

This section covers stages 1 and 2, scoped to a single collection: **NT1**. We'll pull every item in the collection along with metadata (`title`, `region`, `dialect`, `languages`), then look inside each item's `essences` for `.eaf` (ELAN transcript) files, download them, extract their text, and save everything locally.

In [7]:
# %pip install -q pypdf

import io
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

### Fetching Every Item in a Collection

The `items` query is paginated - each response only gives us a page of `results`, plus a `next_page` value we can follow. To get *all* items in a collection, we loop: request a page, collect its results, and move on to `next_page` until it comes back `null`.

In [8]:
def fetch_collection_items(collection_prefix, session, api_url=API_URL):
    """Fetch every item under a collection prefix (e.g. 'MMT1'), paginating through all results."""
    query = '''
        query($full_identifier: String, $page: Int) {
            items(full_identifier: $full_identifier, page: $page) {
                total
                next_page
                results {
                    full_identifier
                    title
                    region
                    dialect
                    access_condition_name
                    collection {
                        identifier
                        title
                    }
                    content_languages {
                        name
                    }
                    essences {
                        filename
                        permalink
                    }
                }
            }
        }
    '''

    all_items = []
    page = 1
    total = None
    while True:
        variables = {'full_identifier': collection_prefix, 'page': page}
        response = session.post(api_url, json={'query': query, 'variables': variables})
        payload = response.json()
        if 'errors' in payload:
            raise RuntimeError(f"GraphQL error on page {page}: {payload['errors']}")
        data = payload['data']['items']
        total = data['total']
        all_items.extend(data['results'])
        if data['next_page'] is None:
            break
        page = data['next_page']

    print(f"Fetched {len(all_items)} items for collection '{collection_prefix}' (API reported total: {total})")
    return all_items

### Finding the Readable Files

Most PARADISEC `essences` are audio, video, or image files - not useful for a text-based RAG system. What we actually want are **ELAN transcripts**: `.eaf` files, an XML annotation format produced by the [ELAN](https://archive.mpi.nl/tla/elan) tool, containing the time-aligned transcription/translation of a recording. We ignore `.jpg`, `.mp3`, `.wav`, etc. for now.

In [ ]:
TEXT_EXTENSIONS = ('.eaf',)

def filter_text_essences(item):
    """Return only the essences on an item whose filename is an ELAN .eaf file."""
    return [e for e in item['essences'] if e['filename'].lower().endswith(TEXT_EXTENSIONS)]

### Getting a Real Download URL: OIDC Login via Playwright

Plain `.txt`/`.pdf` `permalink`s turn out not to serve raw file bytes: the modern PARADISEC front-end (`catalog.paradisec.org.au`, a Vue app called "Oni") uses OpenID Connect (OIDC) login, separate from the simple cookie session our earlier `session.post(.../users/sign_in)` call sets up. The actual file content is served as a **presigned S3 URL**, handed out by `admin-catalog.paradisec.org.au/api/v1/oni/file/<encoded permalink>`, which requires an OAuth `Bearer` access token - not just our cookie.

That access token only exists in the browser's `localStorage` after a real OIDC login completes. So we use [Playwright](https://playwright.dev/python/) to open an actual, visible browser, have you complete the login there yourself (the app's onboarding/login flow has icon-only buttons and multiple steps that aren't worth automating), then pull the `access_token` out of `localStorage` once it appears and attach it to our existing `requests.Session` as a `Bearer` header. After that, plain `requests` calls work for every file - no browser needed per-download.

In [ ]:
# One-time setup: install the actual browser binary Playwright will drive.
# (You've already `pip install`ed the playwright package - this is the separate
# step that downloads Chromium itself.)
!playwright install chromium

In [ ]:
import asyncio
import json

from playwright.async_api import async_playwright


async def get_oidc_access_token(email, manual_timeout=180):
    """Open a real, visible browser to catalog.paradisec.org.au and wait for
    you to complete the login manually - dismiss any onboarding/continue
    screens, click Login, sign in with your credentials. We don't try to
    automate those clicks (the app has icon-only buttons and possibly a
    multi-step onboarding carousel, making selectors too fragile to guess
    reliably). Instead we just poll localStorage in the background until
    oidc-client-ts stores your tokens there, then pull out the access_token.

    IMPORTANT: leave the browser window open until this prints
    'Login detected' - closing it early will raise an error.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        await page.goto('https://catalog.paradisec.org.au/')

        print('A browser window has opened.')
        print(f'Please complete login there manually (sign in as {email}).')
        print('Leave the window open until this cell finishes.')
        print(f'Waiting up to {manual_timeout}s for login to complete...')

        oidc_user = None
        elapsed = 0
        poll_interval = 2
        while elapsed < manual_timeout:
            try:
                local_storage = await page.evaluate('''() => {
                    const items = {};
                    for (let i = 0; i < window.localStorage.length; i++) {
                        const key = window.localStorage.key(i);
                        items[key] = window.localStorage.getItem(key);
                    }
                    return items;
                }''')
                oidc_key = next((k for k in local_storage if k.startswith('oidc.user:')), None)
                if oidc_key is not None:
                    oidc_user = json.loads(local_storage[oidc_key])
                    break
                await page.wait_for_timeout(poll_interval * 1000)
                elapsed += poll_interval
            except Exception as e:
                if 'closed' in str(e).lower():
                    raise RuntimeError(
                        'The browser window was closed before login finished. '
                        'Re-run this cell and leave the window open until "Login detected" prints.'
                    ) from e
                # Otherwise likely a transient mid-navigation hiccup - just retry.
                await asyncio.sleep(poll_interval)
                elapsed += poll_interval

        await browser.close()

    if oidc_user is None:
        raise RuntimeError(
            f'Timed out after {manual_timeout}s waiting for login '
            '(no "oidc.user:" key appeared in localStorage).'
        )
    if 'access_token' not in oidc_user:
        raise RuntimeError(f"No 'access_token' field in stored OIDC user. Keys present: {list(oidc_user.keys())}")

    print('Login detected - access token acquired.')
    return oidc_user['access_token']

In [ ]:
# A visible Chromium window will open - complete the login there yourself
# (dismiss any onboarding screens, click through to sign in) and leave the
# window open. The script detects it automatically once your tokens land in
# localStorage.
access_token = await get_oidc_access_token(email)
session.headers['Authorization'] = f'Bearer {access_token}'
print(f'Access token acquired ({len(access_token)} chars), attached to session.')

### Parsing ELAN (.eaf) Transcripts

An `.eaf` file is XML. Annotations are organized into `TIER`s - typically one tier per transcription line, translation, or speaker - and each `TIER` contains `ANNOTATION` elements whose actual text lives in an `ANNOTATION_VALUE`. Rather than flattening every annotation into one undifferentiated blob, we walk each tier in document order and label every extracted line with its `TIER_ID`, so a transcription tier and a translation tier (for example) stay distinguishable in the resulting text.

In [ ]:
def extract_eaf_text(xml_content):
    """Extract a readable transcript from an ELAN (.eaf) annotation file.

    Walks each TIER in document order and pulls out every ANNOTATION_VALUE,
    labelling each line with its TIER_ID so the transcript keeps that
    structure (e.g. transcription vs. translation vs. per-speaker tiers)
    instead of collapsing into one undifferentiated blob.
    """
    soup = BeautifulSoup(xml_content, 'xml')
    lines = []
    for tier in soup.find_all('TIER'):
        tier_id = tier.get('TIER_ID', 'UNKNOWN_TIER')
        for value in tier.find_all('ANNOTATION_VALUE'):
            text = value.get_text(strip=True)
            if text:
                lines.append(f'{tier_id}: {text}')
    return '\n'.join(lines)

### Downloading and Extracting Text

Each essence's `permalink` points at the actual file in PARADISEC's repository. We download it through our already-authenticated `session` (so this also works for items with a Closed `access_condition`, as long as our account has access), then extract plain text from it: decode `.txt` files directly, and pull text page-by-page out of `.pdf` files with `pypdf`.

If a download or extraction fails, we don't stop the whole run - we record the failure in the manifest and move on.

In [ ]:
import urllib.parse


def download_and_extract(essence, session):
    """Download an essence file and extract its plain text.

    Goes through PARADISEC's Oni API to get a presigned S3 URL (requires our
    OIDC Bearer token on `session`), then fetches the actual bytes from S3.

    Returns a dict with keys: status ('ok'/'error'), text, char_count, error_message.
    """
    filename = essence['filename']
    permalink = essence['permalink']

    encoded_permalink = urllib.parse.quote(permalink, safe='')
    encoded_filename = urllib.parse.quote(filename)
    oni_url = (
        f'https://admin-catalog.paradisec.org.au/api/v1/oni/file/{encoded_permalink}'
        f'?disposition=inline&filename={encoded_filename}&noRedirect=true'
    )

    try:
        # Step 1: ask the Oni API (needs our Bearer token) for a presigned S3 URL.
        presign_response = session.get(oni_url)
        presign_response.raise_for_status()
        payload = presign_response.json()
        presigned_url = payload.get('location') or payload.get('Location')
        if not presigned_url:
            return {
                'status': 'error', 'text': '', 'char_count': 0,
                'error_message': f'No "location" field in oni/file response: {presign_response.text[:300]}',
            }

        # Step 2: fetch the actual bytes from S3 directly. Plain request (not
        # our session) - the URL's own signature is its authorization, and our
        # Bearer header isn't relevant (or wanted) here.
        file_response = requests.get(presigned_url)
        file_response.raise_for_status()
    except Exception as e:
        return {'status': 'error', 'text': '', 'char_count': 0, 'error_message': str(e)}

    content_type = file_response.headers.get('Content-Type', '')
    if 'html' in content_type.lower():
        return {
            'status': 'error', 'text': '', 'char_count': 0,
            'error_message': f'Unexpected HTML content type from presigned URL: {content_type}',
        }

    try:
        if filename.lower().endswith('.eaf'):
            text = extract_eaf_text(file_response.content)
        elif filename.lower().endswith('.pdf'):
            reader = PdfReader(io.BytesIO(file_response.content))
            text = '\n'.join(page.extract_text() or '' for page in reader.pages)
        else:
            try:
                text = file_response.content.decode('utf-8')
            except UnicodeDecodeError:
                text = file_response.content.decode('latin-1')
    except Exception as e:
        return {'status': 'error', 'text': '', 'char_count': 0, 'error_message': f'Extraction failed: {e}'}

    return {'status': 'ok', 'text': text, 'char_count': len(text), 'error_message': ''}

### Putting It Together: Building the NT1 Corpus

Now we run the full pipeline for the `NT1` collection: fetch all items, filter each item's essences down to `.eaf` files, download and extract each one, write the extracted text to a local file under `nt1_sources/<full_identifier>/`, and record a row per essence in `manifest_rows` (one row per sub-item, alongside its parent item's metadata).

In [ ]:
nt1_items = fetch_collection_items('NT1', session)

sources_dir = Path('nt1_sources')
manifest_rows = []
items_with_no_text_essences = 0

for item in nt1_items:
    text_essences = filter_text_essences(item)
    if not text_essences:
        items_with_no_text_essences += 1
        continue

    languages = ', '.join(lang['name'] for lang in item['content_languages'])
    item_dir = sources_dir / item['full_identifier']
    print(f"{item['full_identifier']} — {item['title']} ({len(text_essences)} eaf essence(s))")

    # Metadata header prepended to every essence's saved/stored text - title in
    # particular is often more descriptive than the transcript itself, and this
    # keeps it searchable once these chunks go into Chroma.
    metadata_header = (
        f"Title: {item['title']}\n"
        f"Region: {item['region']}\n"
        f"Dialect: {item['dialect']}\n"
        f"Languages: {languages}\n"
        f"Collection: {item['collection']['identifier']}\n"
        f"Identifier: {item['full_identifier']}\n"
        f"\n"
    )

    for essence in text_essences:
        result = download_and_extract(essence, session)

        local_text_path = ''
        enriched_text = ''
        if result['status'] == 'ok':
            enriched_text = metadata_header + result['text']
            item_dir.mkdir(parents=True, exist_ok=True)
            out_path = item_dir / f"{essence['filename']}.txt"
            out_path.write_text(enriched_text, encoding='utf-8')
            local_text_path = str(out_path)

        manifest_rows.append({
            'collection_id': item['collection']['identifier'],
            'full_identifier': item['full_identifier'],
            'title': item['title'],
            'region': item['region'],
            'dialect': item['dialect'],
            'languages': languages,
            'access_condition_name': item['access_condition_name'],
            'essence_filename': essence['filename'],
            'essence_permalink': essence['permalink'],
            'download_status': result['status'],
            'error_message': result['error_message'],
            'char_count': result['char_count'],  # raw extracted transcript length (diagnostic)
            'local_text_path': local_text_path,
            'text': enriched_text,  # metadata header + transcript - this is what goes into Chroma
        })

        status_icon = '✓' if result['status'] == 'ok' else '✗'
        detail = f" — {result['error_message']}" if result['error_message'] else ''
        print(f"  {status_icon} {essence['filename']} ({result['char_count']} chars){detail}")

print(f"\n{len(nt1_items) - items_with_no_text_essences} of {len(nt1_items)} items had at least one eaf essence; "
      f"{items_with_no_text_essences} had none.")

### Writing the Manifest CSV

The `manifest_rows` list has one row per text essence, carrying its parent item's metadata (`collection_id`, `full_identifier`, `title`, `region`, `dialect`, `languages`) alongside the essence-level fields and the extracted `text` itself. We save this as `nt1_metadata.csv` - this will be our source of truth when we build the Chroma vector database in the next stage.

In [ ]:
nt1_manifest_df = pd.DataFrame(manifest_rows)
nt1_manifest_df.to_csv('nt1_metadata.csv', index=False)
nt1_manifest_df

In [ ]:
ok_count = (nt1_manifest_df['download_status'] == 'ok').sum()
error_count = (nt1_manifest_df['download_status'] == 'error').sum()
total_chars = nt1_manifest_df.loc[nt1_manifest_df['download_status'] == 'ok', 'char_count'].sum()

print('=' * 50)
print('NT1 Text Extraction Summary')
print('=' * 50)
print(f'Items in collection:    {len(nt1_items)}')
print(f'Items with eaf essences: {len(nt1_items) - items_with_no_text_essences}')
print(f'Essences attempted:      {len(nt1_manifest_df)}')
print(f'  Successful:            {ok_count}')
print(f'  Failed:                {error_count}')
print(f'Total characters extracted: {total_chars:,}')
print(f'\nManifest written to: nt1_metadata.csv')
print(f'Text files written under: nt1_sources/')

***
## Stage 3: Vectorizing NT1 into Chroma

Now we take `nt1_metadata.csv` and turn it into a searchable vector database, following the same incremental-update pattern as the [TML Parser Chroma Builder](https://github.com/RichardFreedman/theory_llm) reference notebook:

- **Configuration tracking** (`db_config.json`) - only rebuilds the database if the embedding model or chunk size changes
- **Content-hash tracking** (`content_hashes.json`) - each essence's extracted `text` is hashed; unchanged essences are skipped on re-runs, changed ones are deleted and re-added
- **Deterministic chunk IDs** - so re-running never creates duplicate chunks
- One row of `nt1_metadata.csv` (an essence's `text`, already including the title/region/dialect/languages header) is our unit of "source document" - analogous to one XML file in the TML notebook, except we split it from the manifest CSV rather than re-parsing files from disk.

In [ ]:
import hashlib
import json
import shutil

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

### OpenAI API Key and Database Configuration

We use OpenAI's `text-embedding-3-large` model to convert text into vectors for semantic search - same model as the reference notebook.

In [ ]:
import os

print('Please enter your OpenAI API key:')
openai_api_key = getpass.getpass('API Key: ')
os.environ['OPENAI_API_KEY'] = openai_api_key

if openai_api_key:
    print(f'API key set: {openai_api_key[:7]}...{openai_api_key[-4:]}')
else:
    print('No API key entered')

In [ ]:
# Configuration for the database schema - passed to all the components below.
DB_CONFIG = {
    'version': '1.0',
    'embedding_model': 'text-embedding-3-large',
    'chunk_size': 2000,
    'chunk_overlap': 300,
    'collection_name': 'NT1_eaf',
}

db_path = Path('./chroma-db_nt1')
config_path = db_path / 'db_config.json'

should_recreate = False

if db_path.exists() and config_path.exists():
    with open(config_path, 'r') as f:
        existing_config = json.load(f)

    if (existing_config.get('embedding_model') != DB_CONFIG['embedding_model'] or
        existing_config.get('chunk_size') != DB_CONFIG['chunk_size']):
        print('Breaking changes detected:')
        print(f'   Old: {existing_config}')
        print(f'   New: {DB_CONFIG}')
        should_recreate = True
    else:
        print('Using existing database - configuration unchanged')
        print('Will perform incremental updates only')
elif not db_path.exists():
    print(f'Creating new database at {db_path}')
    should_recreate = True
else:
    print('Database exists but no config found - will recreate')
    should_recreate = True

# Deleting the directory also clears content_hashes.json, so nothing gets
# incorrectly skipped against a database that no longer has it.
if should_recreate and db_path.exists():
    shutil.rmtree(db_path)
    print(f'Deleted existing database at {db_path}')

db_path.mkdir(exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(DB_CONFIG, f, indent=2)

embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

vector_store = Chroma(
    collection_name=DB_CONFIG['collection_name'],
    embedding_function=embeddings,
    persist_directory=str(db_path),
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=DB_CONFIG['chunk_size'],
    chunk_overlap=DB_CONFIG['chunk_overlap'],
    length_function=len,
    is_separator_regex=False,
)

### Functions to Process the Manifest and Update the Chroma DB

Each essence (one CSV row) gets a `source` id of `<full_identifier>/<essence_filename>`. We hash its `text` to detect changes between runs: unchanged essences are skipped, changed ones have their old chunks deleted before new ones are added, and brand-new essences are simply added. Chunk IDs are deterministic (`md5(source_chunk_index)`), so re-running never creates duplicates.

In [ ]:
def get_content_hash(text):
    """MD5 hash of an essence's text, to detect changes between runs."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def load_content_hashes():
    hash_file = db_path / 'content_hashes.json'
    if hash_file.exists():
        with open(hash_file, 'r') as f:
            return json.load(f)
    return {}


def save_content_hashes(hashes):
    hash_file = db_path / 'content_hashes.json'
    with open(hash_file, 'w') as f:
        json.dump(hashes, f, indent=2)


def generate_chunk_id(source_id, chunk_index):
    """Deterministic ID for a chunk, so re-running never creates duplicates."""
    id_string = f'{source_id}_chunk_{chunk_index}'
    return hashlib.md5(id_string.encode()).hexdigest()


def process_manifest(csv_path='nt1_metadata.csv', force_reprocess=False):
    """
    Read the manifest CSV and add/update its essences in the Chroma DB.

    Args:
        csv_path: path to the manifest CSV (one row per essence)
        force_reprocess: if True, re-chunk and re-embed every essence
            regardless of whether its content hash has changed
    """
    df = pd.read_csv(csv_path)
    df = df[df['download_status'] == 'ok'].copy()

    existing_hashes = load_content_hashes()
    new_hashes = {}

    total_chunks = 0
    essences_new = 0
    essences_updated = 0
    essences_skipped = 0

    for _, row in df.iterrows():
        text = row['text']
        if not isinstance(text, str) or not text.strip():
            continue

        source_id = f"{row['full_identifier']}/{row['essence_filename']}"
        current_hash = get_content_hash(text)
        new_hashes[source_id] = current_hash

        already_seen = source_id in existing_hashes
        hash_changed = already_seen and existing_hashes[source_id] != current_hash

        if already_seen and not hash_changed and not force_reprocess:
            essences_skipped += 1
            continue

        if already_seen and (hash_changed or force_reprocess):
            try:
                vector_store.delete(where={'source': source_id})
            except Exception as e:
                print(f'  Note: could not delete old chunks for {source_id}: {e}')
            essences_updated += 1
        else:
            essences_new += 1

        metadata = {
            'collection_id': str(row['collection_id']),
            'full_identifier': str(row['full_identifier']),
            'title': str(row['title']) if pd.notna(row['title']) else '',
            'region': str(row['region']) if pd.notna(row['region']) else '',
            'dialect': str(row['dialect']) if pd.notna(row['dialect']) else '',
            'languages': str(row['languages']) if pd.notna(row['languages']) else '',
            'access_condition_name': str(row['access_condition_name']) if pd.notna(row['access_condition_name']) else '',
            'essence_filename': str(row['essence_filename']),
            'essence_permalink': str(row['essence_permalink']),
            'source': source_id,
        }

        chunks = text_splitter.create_documents(texts=[text], metadatas=[metadata])
        chunk_ids = [generate_chunk_id(source_id, i) for i in range(len(chunks))]

        vector_store.add_documents(documents=chunks, ids=chunk_ids)
        total_chunks += len(chunks)
        print(f"  {'+' if source_id not in existing_hashes else '↻'} {source_id} - {len(chunks)} chunk(s)")

    save_content_hashes(new_hashes)

    print(f'\n{"=" * 50}')
    print('Chroma Processing Complete')
    print(f'{"=" * 50}')
    print(f'Essences in manifest:  {len(df)}')
    print(f'  New:                 {essences_new}')
    print(f'  Updated:             {essences_updated}')
    print(f'  Skipped (unchanged): {essences_skipped}')
    print(f'Total chunks added:    {total_chunks}')

### Run It

Only new or changed essences get (re-)embedded and (re-)added; unchanged ones are skipped. Use `force_reprocess=True` to rebuild everything regardless of the content hash.

In [ ]:
process_manifest('nt1_metadata.csv', force_reprocess=False)

In [ ]:
# Helper functions for database management.

def get_db_stats():
    """Print and return summary statistics about the current database."""
    all_docs = vector_store.get()

    if not all_docs or not all_docs.get('metadatas'):
        print('Database is empty')
        return

    total_docs = len(all_docs['ids'])
    titles = set()
    sources = set()
    languages = set()

    for metadata in all_docs['metadatas']:
        if metadata:
            titles.add(metadata.get('title', ''))
            sources.add(metadata.get('source', ''))
            if metadata.get('languages'):
                languages.add(metadata['languages'])

    print('Database Statistics:')
    print(f'  Total chunks: {total_docs}')
    print(f'  Unique essences (sources): {len(sources)}')
    print(f'  Unique item titles: {len(titles)}')
    print(f'  Distinct language groupings: {len(languages)}')

    return {'total_docs': total_docs, 'sources': sorted(sources), 'titles': sorted(titles)}


def delete_source_essence(full_identifier, essence_filename):
    """Delete all chunks belonging to one essence."""
    source_id = f'{full_identifier}/{essence_filename}'
    try:
        vector_store.delete(where={'source': source_id})
        print(f'Deleted all chunks for {source_id}')
        hashes = load_content_hashes()
        if source_id in hashes:
            del hashes[source_id]
            save_content_hashes(hashes)
    except Exception as e:
        print(f'Error deleting {source_id}: {e}')


get_db_stats()

### Try a Semantic Search

Search by meaning, not just keyword match - edit `query` below to try your own.

In [ ]:
query = 'traditional ceremony and song'

results = vector_store.similarity_search_with_score(query, k=5)

print(f"Query: '{query}'\n")
for i, (doc, score) in enumerate(results, 1):
    print(f'{"=" * 60}')
    print(f'Result {i} - Similarity Score: {score:.4f}')
    print(f"  Title: {doc.metadata.get('title', 'N/A')}")
    print(f"  Identifier: {doc.metadata.get('full_identifier', 'N/A')}")
    print(f"  Essence: {doc.metadata.get('essence_filename', 'N/A')}")
    print(f"  Region: {doc.metadata.get('region', 'N/A')} | Dialect: {doc.metadata.get('dialect', 'N/A')}")
    preview = doc.page_content[:400] + '...' if len(doc.page_content) > 400 else doc.page_content
    print(f'\n  {preview}')
    print()